# Notebook 08: Audio - Text-to-Speech (TTS)

**Learning Objectives:**
- Generate speech from text using TTS models
- Use SpeechT5 for natural-sounding speech
- Control voice characteristics with speaker embeddings
- Save and play generated audio

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|--------------|------------|------|---------|-------------------|-------|
| **CPU/GPU** | microsoft/speecht5_tts | 200MB | 4GB | 6GB VRAM (RTX 4080) | Good quality |

### Software Requirements
- Python 3.8+
- Libraries: `transformers`, `torch`, `soundfile`, `datasets`

In [ ]:
import sys
import random
import time

import numpy as np
import torch
import transformers
from transformers import (
    SpeechT5Processor,
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan,
    pipeline,
)
from datasets import load_dataset
import soundfile as sf
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Version metadata
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Expected Behaviors

### First Time Running
- **Model Download**: ~200MB for SpeechT5 TTS (~2-3 minutes)
- Also downloads vocoder (~50MB)
- Downloads speaker embeddings dataset (~100MB)
- Total: ~350MB first run

### Setup Cell Output
```
PyTorch version: 2.x.x
CUDA available: True/False
```

### Model Loading
```
Model loaded on: cpu (or cuda)
Speaker embeddings loaded
```
- **CPU**: 10-15 seconds (loads 3 components)
- **GPU**: 5-8 seconds

### Audio Output
- Saves as WAV file at 16kHz sample rate
- File size: ~1.5MB per minute of audio
- Can be played in Jupyter with `IPython.display.Audio()`

### Speech Quality
- **Natural sounding** for most English text
- **Intonation**: Reasonably natural, slight robotic quality
- **Pronunciation**: 90-95% accurate for common words
- **Prosody**: Handles punctuation (pauses, emphasis)

### Voice Characteristics
- Controlled by speaker embeddings
- Different embeddings = different voice characteristics
- Current notebooks use single voice (can experiment with others)
- Voice quality is consistent across generations

### Performance
- **Short sentence** (10-20 words):
  - CPU: 3-5 seconds
  - GPU: 1-2 seconds
- **Long paragraph** (100+ words):
  - CPU: 15-25 seconds
  - GPU: 5-8 seconds

### Text Limitations
- **Works best with**: Clear, grammatical English
- **Maximum length**: ~200 words per generation (split longer text)
- **Numbers**: Speaks digits individually (e.g., "1234" → "one two three four")
- **Abbreviations**: May mispronounce (expand them first)

### Common Issues
- **Monotone speech**: Try different speaker embeddings
- **Mispronunciations**: Phonetic spelling may help
- **Unnatural pauses**: Check punctuation
- **Cutoff**: Text too long, split into chunks

### Expected Output Files
- `output.wav`, `hello.wav`, etc.
- Saved in notebook directory
- Can open with any audio player

### Quality Factors
- Punctuation affects pacing and intonation
- ALL CAPS may sound overly emphasized
- Question marks add rising intonation

## Overview

**Text-to-Speech (TTS)** converts written text into natural-sounding spoken audio. Modern neural TTS systems have largely replaced older concatenative and parametric approaches, producing speech that is often difficult to distinguish from human recordings.

### How SpeechT5 Works

SpeechT5 uses an **encoder-decoder architecture** inspired by the T5 text model:

1. **Text Encoder**: Converts input text into a sequence of hidden representations, capturing phonetic and linguistic information
2. **Decoder**: Transforms these representations into a mel-spectrogram (a visual representation of audio frequencies over time)
3. **Vocoder (HiFi-GAN)**: Converts the mel-spectrogram into an actual audio waveform

### Speaker Embeddings

SpeechT5 uses **x-vector speaker embeddings** to control voice characteristics. These embeddings are fixed-size vectors that capture a speaker's vocal identity (pitch, timbre, speaking style). By swapping embeddings, you can generate the same text in different voices without retraining the model.

### Key Concepts

- **Mel-spectrogram**: A time-frequency representation of audio, used as an intermediate format between text and waveform
- **Vocoder**: A model that converts spectrograms back into audible waveforms
- **Sample rate**: Number of audio samples per second (16kHz for SpeechT5, meaning 16,000 data points per second of audio)

## Setup and Installation

The imports cell above loads all necessary libraries. SpeechT5 requires three components: a **processor** (tokenizes text), a **model** (generates mel-spectrograms), and a **vocoder** (converts spectrograms to audio). We also need the `datasets` library to load pre-computed speaker embeddings.

## Model Selection

SpeechT5 is a unified model that works well for both CPU and GPU. It is relatively small (~200MB) and produces good-quality English speech. There is only one model variant, but the vocoder component can be swapped for different audio quality tradeoffs.

## Method 1: Pipeline API

The Pipeline API is the simplest way to use TTS models. It handles model loading, text processing, and audio generation in a single call. Note that the TTS pipeline requires speaker embeddings for SpeechT5, which we load from a pre-computed dataset.

In [ ]:
# Load speaker embeddings (needed for both methods)
print("Loading speaker embeddings...")
try:
    embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
    speaker_embeddings = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
    print(f"Speaker embeddings loaded (shape: {speaker_embeddings.shape})")
except Exception as e:
    print(f"Error loading embeddings: {e}")
    print("Ensure you have internet access for first-time download.")

In [ ]:
# Method 1: Pipeline API — simplest approach
print("Creating TTS pipeline...")
try:
    tts_pipeline = pipeline(
        "text-to-speech",
        model=MODEL_NAME,
        vocoder=VOCODER_NAME,
        device=device,
    )
    print("TTS pipeline ready")
except Exception as e:
    print(f"Error creating pipeline: {e}")
    tts_pipeline = None

In [ ]:
def generate_speech_pipeline(
    text: str,
    output_file: str = "output_pipeline.wav",
) -> np.ndarray:
    """Generate speech from text using the Pipeline API.

    Args:
        text: Input text to convert to speech.
        output_file: Path to save the generated WAV file.

    Returns:
        NumPy array of audio samples.
    """
    result = tts_pipeline(
        text,
        forward_params={"speaker_embeddings": speaker_embeddings},
    )
    audio_array = np.array(result["audio"].squeeze())
    sample_rate = result["sampling_rate"]

    sf.write(output_file, audio_array, samplerate=sample_rate)
    print(f"Audio saved to: {output_file} ({len(audio_array)} samples at {sample_rate}Hz)")
    return audio_array


# Generate speech with the pipeline
audio = generate_speech_pipeline(
    "Hello, welcome to the HuggingFace tutorial on text-to-speech!",
    "hello_pipeline.wav",
)
print(f"Duration: {len(audio) / 16000:.1f}s")

## Method 2: Manual Model Loading

For more control over the generation process, we can load each component individually. This approach lets you adjust generation parameters, swap vocoders, and manage GPU memory more precisely.

In [ ]:
# Load components manually
print("Loading model components manually...")
processor = SpeechT5Processor.from_pretrained(MODEL_NAME)
tts_model = SpeechT5ForTextToSpeech.from_pretrained(MODEL_NAME).to(device)
tts_vocoder = SpeechT5HifiGan.from_pretrained(VOCODER_NAME).to(device)

# Move speaker embeddings to device for manual usage
speaker_embeddings_device = speaker_embeddings.to(device)
print(f"All components loaded on: {device}")

In [ ]:
def text_to_speech_manual(
    text: str,
    output_file: str = "output.wav",
    speaker_emb: torch.Tensor | None = None,
) -> np.ndarray:
    """Convert text to speech using manual model components.

    Args:
        text: Input text to convert to speech.
        output_file: Path to save the generated WAV file.
        speaker_emb: Speaker embedding tensor. Uses default if None.

    Returns:
        NumPy array of audio samples.
    """
    if speaker_emb is None:
        speaker_emb = speaker_embeddings_device

    inputs = processor(text=text, return_tensors="pt").to(device)

    with torch.no_grad():
        speech = tts_model.generate_speech(
            inputs["input_ids"],
            speaker_emb,
            vocoder=tts_vocoder,
        )

    speech_np = speech.cpu().numpy()
    sf.write(output_file, speech_np, samplerate=16000)
    print(f"Audio saved to: {output_file}")
    return speech_np


# Test manual generation
audio = text_to_speech_manual(
    "This is generated using the manual model loading approach.",
    "hello_manual.wav",
)
print(f"Generated {len(audio)} samples ({len(audio) / 16000:.1f}s)")

## Practical Applications

Now that we have both pipeline and manual approaches working, let's explore real-world use cases: generating speech for multiple phrases, experimenting with different speaker voices, and processing longer text by splitting it into chunks.

In [ ]:
# Application 1: Batch generation for multiple phrases
phrases = [
    "Artificial intelligence is transforming the world.",
    "Machine learning models can now understand and generate speech.",
    "This technology has many practical applications.",
]

print("=== BATCH SPEECH GENERATION ===")
for i, phrase in enumerate(phrases, 1):
    output_file = f"phrase_{i}.wav"
    audio = text_to_speech_manual(phrase, output_file)
    duration = len(audio) / 16000
    print(f"  {i}. '{phrase[:50]}...' -> {duration:.1f}s")

In [ ]:
def try_different_voices(
    text: str,
    num_voices: int = 3,
) -> list[np.ndarray]:
    """Generate speech with different speaker voices.

    Args:
        text: Input text to convert to speech.
        num_voices: Number of different voices to try.

    Returns:
        List of audio arrays, one per voice.
    """
    audios = []
    print(f"=== GENERATING WITH {num_voices} DIFFERENT VOICES ===")

    for i in range(min(num_voices, len(embeddings_dataset))):
        speaker_emb = torch.tensor(
            embeddings_dataset[i]["xvector"]
        ).unsqueeze(0).to(device)

        inputs = processor(text=text, return_tensors="pt").to(device)

        with torch.no_grad():
            speech = tts_model.generate_speech(
                inputs["input_ids"],
                speaker_emb,
                vocoder=tts_vocoder,
            )

        speech_np = speech.cpu().numpy()
        output_file = f"voice_{i + 1}.wav"
        sf.write(output_file, speech_np, samplerate=16000)
        audios.append(speech_np)
        print(f"  Voice {i + 1}: {len(speech_np)} samples ({len(speech_np) / 16000:.1f}s) -> {output_file}")

    return audios


# Application 2: Compare different speaker voices
audios = try_different_voices(
    "This is a test of different voice characteristics.",
    num_voices=3,
)

## Performance Benchmarking

Let's measure generation speed across different text lengths to understand the relationship between input size and inference time. This helps set expectations for real-world applications.

In [ ]:
def benchmark_tts(
    texts: list[str],
    labels: list[str],
) -> pd.DataFrame:
    """Benchmark TTS generation speed across different text lengths.

    Args:
        texts: List of input texts to benchmark.
        labels: Descriptive labels for each text.

    Returns:
        DataFrame with benchmark results.
    """
    results = []

    for text, label in zip(texts, labels):
        word_count = len(text.split())
        start_time = time.time()
        inputs = processor(text=text, return_tensors="pt").to(device)
        with torch.no_grad():
            speech = tts_model.generate_speech(
                inputs["input_ids"],
                speaker_embeddings_device,
                vocoder=tts_vocoder,
            )
        elapsed = time.time() - start_time
        audio_duration = len(speech) / 16000

        results.append({
            "Text": label,
            "Words": word_count,
            "Gen Time (s)": round(elapsed, 1),
            "Audio Duration (s)": round(audio_duration, 1),
            "Real-Time Factor": round(elapsed / audio_duration, 2) if audio_duration > 0 else 0,
            "Device": str(device),
        })

    return pd.DataFrame(results)


benchmark_texts = [
    "Hello world.",
    "Artificial intelligence is transforming how we interact with technology.",
    "The quick brown fox jumps over the lazy dog. This sentence contains every letter of the alphabet and is commonly used for testing purposes.",
]
benchmark_labels = ["Short (2 words)", "Medium (9 words)", "Long (25 words)"]

benchmark_df = benchmark_tts(benchmark_texts, benchmark_labels)
print("=== TTS PERFORMANCE BENCHMARK ===")
benchmark_df

## State-of-the-Art Open Models (Not Covered)

While SpeechT5 is a solid foundation for TTS, several cutting-edge models offer superior naturalness, multilingual support, and voice cloning.

| Model | Size | Speed | Quality | Languages | Special Features |
|-------|------|-------|---------|-----------|------------------|
| **SpeechT5** | 200MB | Fast | Good | English | Learning-friendly |
| **VITS** | 400MB | Very Fast | Great | Multi | Production-ready, no vocoder needed |
| **YourTTS** | 500MB | Fast | Good | 16+ | Zero-shot voice cloning |
| **XTTS v2** | 1.8GB | Medium | Excellent | 16 | Voice cloning from 6s of audio |
| **Bark** | 2.8GB | Slow | Great | Multi | Music, emotions, sound effects |

These models require 8-16GB VRAM and are best explored once you're comfortable with the basics covered here. For more details, see [Papers With Code - Speech Synthesis](https://paperswithcode.com/task/speech-synthesis).

## Exercises

1. **Long Text**: Generate speech for a paragraph (100+ words)
2. **Voice Comparison**: Try all available speaker embeddings and compare
3. **Custom Text**: Generate speech for your own text
4. **Punctuation Effect**: Test how punctuation affects speech prosody
5. **Multiple Languages**: Does the model handle non-English text?

In [ ]:
# Exercise starter: Generate speech for a longer paragraph
long_text = (
    "Text to speech technology has come a long way in recent years. "
    "Modern neural models can produce speech that is nearly "
    "indistinguishable from human recordings. These advances have "
    "enabled applications in accessibility, content creation, and "
    "interactive voice assistants."
)
long_audio = text_to_speech_manual(long_text, "exercise_long.wav")
print(f"Long text: {len(long_text.split())} words -> {len(long_audio) / 16000:.1f}s audio")

## Key Takeaways

✅ **SpeechT5** generates natural-sounding speech

✅ **Speaker embeddings** control voice characteristics

✅ **Vocoder** (HiFiGAN) converts features to waveform

✅ Generated audio is saved as WAV files at 16kHz

✅ Works well for English text

## Next Steps

- Try **Notebook 09**: Multimodal Image-to-Text
- Explore other TTS models on [HuggingFace Hub](https://huggingface.co/models?pipeline_tag=text-to-speech)
- Learn about voice cloning and custom speaker embeddings

## Resources

- [SpeechT5 Paper](https://arxiv.org/abs/2110.07205)
- [TTS Task Guide](https://huggingface.co/docs/transformers/tasks/text-to-speech)
- [SpeechT5 Model Card](https://huggingface.co/microsoft/speecht5_tts)